# Basic NLP with `text_records()`

This notebook shows a simple handoff from `crategraph` to an NLP library.

The point of using `crategraph` here is that we do not read every file directly. We use the RO-Crate graph to select file entities, keep provenance and basic entity metadata with `text_records()`, then group the NLP results by genre metadata recorded in the graph.

## Install TextBlob

Run this cell once if TextBlob is not already available in your notebook environment.

In [ ]:
!uv pip install textblob

In [ ]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from textblob import TextBlob

from crategraph import Crate

## Load the crate

Load the Australian Corpus of English crate from `data/`.

In [ ]:
crate = Crate(Path("data/ldaca/Australian_Corpus_of_English"))
crate.summary()

In [ ]:
crate.glimpse()

## Select files to analyse

`text_records()` extracts text from file entities. We still make the file selection explicit so the notebook demonstrates the graph-to-NLP handoff on a filtered graph.

In [ ]:
text_files = crate.select(entity_types=["File"])

text_files.summary()

Count the file genres recorded in the graph. The genre relationship is specific to this crate, but the pattern is general: use graph metadata to make NLP groupings meaningful.

In [ ]:
text_file_ids = {entity.id for entity in text_files.entities}

genre_rows = [
    {"entity_id": rel.source, "genre": crate.get(rel.target).name}
    for rel in crate.relationships
    if rel.type == "ldac:linguisticGenre" and rel.source in text_file_ids
]

Counter(row["genre"] for row in genre_rows)

## Hand the text to NLP tools

`text_records()` returns one row-like record per text unit, with provenance columns kept alongside the text. Use `include_properties` to carry selected entity metadata into the same rows.

For a quick demo, take 15 files per genre.

In [ ]:
selected = (
    pd.DataFrame(genre_rows)
    .sort_values(["genre", "entity_id"])
    .groupby("genre", group_keys=False)
    .head(15)
)

records = pd.DataFrame(
    text_files.text_records(
        include_properties=["name"],
        filters={"entity_id": selected["entity_id"].tolist()},
    )
).merge(selected, on="entity_id", how="left")

records[["entity_id", "genre", "name", "text"]].head()

In [ ]:
word_pattern = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")


def word_count(text):
    return len(word_pattern.findall(text))


records["word_count"] = records["text"].map(word_count)
records["polarity"] = records["text"].map(lambda text: TextBlob(text).sentiment.polarity)

records[["entity_id", "genre", "word_count", "polarity", "name"]].head()

## Summarise by graph-derived genre

The grouping column comes from RO-Crate relationships, not from file names.

In [ ]:
records.groupby("genre").agg(
    documents=("entity_id", "count"),
    mean_words=("word_count", "mean"),
    mean_polarity=("polarity", "mean"),
).sort_values("documents", ascending=False)